# MobiHet V4 S
Frozen pilot-v4-mala/support contracts in input. All learned models DAGSNet. No raw/perturbed/latent/per-record gradients exported; trusted cohort sum >=5, not crypto/DP. Stage U only; internal group/old, r20 private/gp; R1/R2 not proven. W&B authenticated online before train, private inline owner-authorized fallback. Ten classification metrics and per-class F1 from mean client confusion, not pooled confusion.

In [ ]:

import os, subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','--quiet','wandb==0.29.0'],check=True)
import wandb
_INLINE_WANDB_KEY = ""
try:
    from kaggle_secrets import UserSecretsClient
    _wandb_key=UserSecretsClient().get_secret('wandb_key')
except Exception:
    _wandb_key=_INLINE_WANDB_KEY
assert _wandb_key, 'W&B key required before training'
assert wandb.login(key=_wandb_key,anonymous='never',relogin=True,verify=True)
os.environ['WANDB_API_KEY']=_wandb_key
del _wandb_key
assert wandb.Api(timeout=30).default_entity, 'W&B online authentication failed'
print('W&B authenticated online',flush=True)


In [ ]:
CODE=['__init__.py', 'archs.py', 'channel.py', 'dagsnet.py', 'metrics.py', 'mobility.py', 'new_fl.py', 'new_methods.py', 'p2.py', 'pilot.py', 'prepare.py', 'v2_math.py', 'v2_methods.py', 'v2_run.py', 'v3_dense.py', 'v3_math.py', 'v3_methods.py', 'v3_run.py', 'v4_mala.py', 'v4_run.py', 'v4_schema.py', 'v4_support.py']

import hashlib,json,shutil,time
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
import torch
count=torch.cuda.device_count()
assert count==2 and all(torch.cuda.get_device_capability(i)==(7,5) for i in range(count))
assert torch.__version__=='2.10.0+cu128',torch.__version__
print('torch',torch.__version__,'gpus',[torch.cuda.get_device_name(i) for i in range(count)],flush=True)
hits=[p.parent for p in Path('/kaggle/input').rglob('v2_run.py') if (p.parent/'train_0_X.npy').exists()]
assert len(hits)==1,hits
SRC=hits[0]; WORK=Path('/kaggle/working/work'); OUT=Path('/kaggle/working/runs')
SOURCE=Path('/kaggle/working/source'); SOURCE.mkdir(exist_ok=True)
for name in ('mh','knowledge','papers/pilot/fixture'): (WORK/name).mkdir(parents=True,exist_ok=True)
manifest=json.loads((SRC/'source_manifest.json').read_text())
for name,sha in manifest.items():
    assert hashlib.sha256((SRC/name).read_bytes()).hexdigest()==sha,('source mismatch',name)
    shutil.copyfile(SRC/name,SOURCE/name)
for f in SRC.iterdir():
    if f.name in CODE: dst=WORK/'mh'/f.name
    elif f.name=='meta.json': dst=WORK/'knowledge'/f.name
    elif f.name.startswith('pilot-v4-') and f.suffix=='.md': continue
    else: dst=WORK/'papers/pilot/fixture'/f.name
    shutil.copyfile(f,dst)
fixture=json.loads((SRC/'manifest.json').read_text())
for name,sha in fixture['files'].items():
    assert hashlib.sha256((SRC/name).read_bytes()).hexdigest()==sha,('fixture mismatch',name)
shutil.copyfile(SRC/'source_manifest.json',SOURCE/'source_manifest.json')
shutil.copyfile(SRC/'manifest.json',SOURCE/'fixture_manifest.json')
print('verified source and fixture',len(manifest),len(fixture['files']),flush=True)
sys.path.insert(0,str(WORK))
import numpy as np
from mh.dagsnet import build_model,CFG
from mh.v2_math import Layout,CountSketch,SeedBasis,unpack
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True
torch.backends.cuda.matmul.allow_tf32=False
for gpu in range(2):
    device=torch.device(f'cuda:{gpu}'); torch.cuda.set_device(device); torch.manual_seed(42)
    model=build_model(dict(CFG,n_features=66,num_classes=16),latent=True).to(device).eval()
    x=torch.from_numpy(np.load(SRC/'train_0_X.npy')[:32].astype('float32')).to(device)
    y=torch.from_numpy(np.load(SRC/'train_0_y.npy')[:32].astype('int64')).to(device)
    layout=Layout(model); assert layout.size==395024
    with torch.autocast('cuda',dtype=torch.float16): z=model(x)[0].float()
    loss=torch.nn.functional.cross_entropy(z,y); assert torch.isfinite(loss); loss.backward()
    gradient=torch.cat([p.grad.float().reshape(-1) for p in model.parameters()])
    assert torch.isfinite(gradient).all()
    sketch=CountSketch(layout.size,3,256,1701,device)
    packet,delta=sketch.patch(sketch.encode(gradient),128,0); assert torch.isfinite(delta).all()
    assert unpack(packet,device,0)['version']==1
    basis=SeedBasis(model,8,1901,device); coeff=torch.zeros(8,device=device,requires_grad=True)
    torch.nn.functional.cross_entropy(basis.forward(model,coeff,x)[0].float(),y).backward()
    assert torch.isfinite(coeff.grad).all()
    cm=torch.bincount(y*16+z.argmax(1),minlength=256); assert int(cm.sum())==len(y)
    print('T4 real-model deterministic preflight PASS',gpu,flush=True)
    del model,x,y,z,gradient,sketch,basis,coeff,delta; torch.cuda.empty_cache()
torch.cuda.set_device(0)

from types import SimpleNamespace
from mh.v4_mala import potential,transition,accept
from mh.v4_support import Method as Support
for gpu in range(2):
    device=torch.device(f'cuda:{gpu}'); torch.cuda.set_device(device)
    model=build_model(dict(CFG,n_features=66,num_classes=16),latent=True).to(device).eval()
    g=torch.Generator(device=device).manual_seed(4701)
    z=torch.randn(16,66,device=device,generator=g); labels=torch.arange(16,device=device)
    mu=torch.zeros(16,66,device=device); sd=torch.ones_like(mu)
    lp,score=potential(model,z,labels,mu,sd)
    proposed=transition(z,lp,score,g); nlp,ngrad=potential(model,proposed,labels,mu,sd)
    next_z,_,_,taken=accept(z,proposed,lp,score,nlp,ngrad,g)
    assert torch.isfinite(next_z).all() and all(p.requires_grad for p in model.parameters())
    x=torch.from_numpy(np.load(SRC/'train_0_X.npy')[:32]).to(device)
    y=torch.from_numpy(np.load(SRC/'train_0_y.npy')[:32]).to(device).long()
    run=SimpleNamespace(dev=device,K=5,X=[x]*5,Y=[y]*5,n=[32]*5,bytes=[],init=model,
                        a=SimpleNamespace(arm='s_structural'))
    support=Support(run); sx=support.draw_support(labels,g,0)
    assert torch.isfinite(sx).all() and len(support.support_packet)==10592
    print('T4 V4 score/Metropolis/support codec preflight PASS',gpu,flush=True)
    del model,z,labels,mu,sd,lp,score,proposed,nlp,ngrad,next_z,taken,x,y,run,support,sx
    torch.cuda.empty_cache()
torch.cuda.set_device(0)


In [ ]:
SEED=42
METHODS=['s_diag', 's_marginal', 's_structural', 's_own', 's_shuffle']

T0=time.time(); OUT.mkdir(exist_ok=True)
queue=list(METHODS); slots=[0,0,1,1]; active={}; done={}; seen={}
while queue or active:
    for slot,gpu in enumerate(slots):
        if slot in active or not queue or time.time()-T0>=3*3600: continue
        arm=queue.pop(0); folder=OUT/f'{arm}_s{SEED}'; folder.mkdir(exist_ok=True)
        env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpu),OMP_NUM_THREADS='1',MKL_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1')
        command=[sys.executable,'-m','mh.v4_run','--arm',arm,'--seed',str(SEED),'--out',str(OUT),
                 '--wandb_online','--max_hours','4']
        log=open(folder/'log.txt','w')
        active[slot]=(arm,subprocess.Popen(command,cwd=WORK,env=env,stdout=log,stderr=subprocess.STDOUT),log)
        print('start',arm,'gpu',gpu,flush=True)
    for slot,(arm,proc,log) in list(active.items()):
        lines=(OUT/f'{arm}_s{SEED}'/'log.txt').read_text().splitlines()
        for line in lines[seen.get(arm,0):]:
            if line.startswith(('setup','r','done','WANDB','Traceback','budget')) or 'Error' in line:
                print(arm,'|',line,flush=True)
        seen[arm]=len(lines)
        if proc.poll() is not None:
            done[arm]=proc.returncode; log.close(); del active[slot]
    if queue and not active and time.time()-T0>=3*3600: break
    if queue or active: time.sleep(20)
summary={arm:{'rc':rc,'complete':json.loads((OUT/f'{arm}_s{SEED}'/'result.json').read_text())['complete']
             if (OUT/f'{arm}_s{SEED}'/'result.json').exists() else False} for arm,rc in done.items()}
(OUT/f'summary_s{SEED}.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary),flush=True)
shutil.rmtree(WORK)  # raw fixture never published as notebook output
needle=os.environ['WANDB_API_KEY'].encode()
for root in (OUT,SOURCE):
  for path in root.rglob('*'):
    if not path.is_file(): continue
    carry=b''
    with path.open('rb') as handle:
        while True:
            part=handle.read(1024*1024)
            if not part: break
            assert needle not in carry+part,('credential in output',path.name)
            carry=(carry+part)[-len(needle):]
assert set(summary)==set(METHODS) and all(v['rc']==0 and v['complete'] for v in summary.values()),summary
